## Interpretación del modelo Seleccionado


### By:
Jhonner Acosta

### Date:
2026-08-21

### Description:

Requerimiento
Analizar las características del modelo obtenido, con el fin de tener un listado de pruebas y experimentos que se van a realizar en la siguiente iteración. Crear un nuevo branch de git (Usar Gitflow).

Tomar como ejemplo los pasos de: https://joserzapata.github.io/post/ciencia-datos-proyecto-python/7-model_interpretation/

En este proceso se incluye :

Interpretar el modelo seleccionado
Identificar los atributos más importantes
Realizar el análisis de Learning curve
obtener las gráficas de escalabilidad con tiempo de entrenamiento y score
¿Cuáles son las consecuencias de las malas predicciones?
¿Qué tipo de errores comete el modelo?
¿A qué se deben los errores?
outliers?
Clase desbalanceada?
¿Errores en el ingreso de datos?
Encoders?
etc
Puede utilizar las librerías o herramientas que considere para resolver la tarea

Entregables
Notebook con los pasos descritos anteriormente y el modelo entrenado en formato .joblib

Se debe realizar un Pull request para ingresar el notebook a la rama main para esto debe tener mínimo 1 revisiones de otras personas del Curso y que pase los check del CI/CD.

## 📚 Import  libraries

In [19]:
from pathlib import Path

import joblib
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import learning_curve, train_test_split

In [ ]:
_root = next(
    p for p in [Path().resolve(), *Path().resolve().parents] if (p / "pyproject.toml").exists()
)
DATA_DIR = _root / "data"
MODELS_DIR = _root / "models"
FIGURES_DIR = _root / "notebooks" / "7-deploy"
FIGURES_DIR.mkdir(exist_ok=True)

SEED = 42
TEST_SIZE = 0.2
CV_FOLDS = 5
N_REPEATS = 10

In [21]:
# Validación de prerequisitos
parquet_path = DATA_DIR / "02_intermediate" / "corazon_type_fixed.parquet"
model_path = MODELS_DIR / "06_svm_model.joblib"

missing = []
if not parquet_path.exists():
    missing.append(
        f"  - {parquet_path.relative_to(_root)} (ejecutar notebook 04.Feature_Engineering)"
    )
if not model_path.exists():
    missing.append(f"  - {model_path.relative_to(_root)} (ejecutar notebook 06.Seleccion_Modelo)")

if missing:
    raise FileNotFoundError("Prerequisitos faltantes:\n" + "\n".join(missing))

print("✓ Todos los prerequisitos están disponibles.")

✓ Todos los prerequisitos están disponibles.


## 💾 Load data

In [22]:
# Cargar datos
df = pd.read_parquet(DATA_DIR / "02_intermediate" / "corazon_type_fixed.parquet")
df = df.drop_duplicates()

TARGET = "disease"
DROP_COLS = ["exang"]

num_cols = ["age", "rest_bp", "chol", "max_hr", "old_peak"]
cat_cols = ["sex", "chest_pain", "fbs", "rest_ecg", "slope", "ca", "thal"]

X = df.drop(columns=[TARGET])
y = df[TARGET].astype(int)

X_clean = X.drop(columns=DROP_COLS, errors="ignore")[num_cols + cat_cols].copy()

for col in cat_cols:
    X_clean[col] = X_clean[col].astype(object).where(X_clean[col].notna(), other=np.nan)
for col in num_cols:
    X_clean[col] = pd.to_numeric(X_clean[col], errors="coerce").astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X_clean, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")

Train: (323, 12) | Test: (81, 12)


## 👷 Cargar modelo

In [23]:
# Cargar pipeline entrenado (feature engineering + SVM)
model_path = MODELS_DIR / "06_svm_model.joblib"
pipeline = joblib.load(model_path)
print(f"Modelo cargado: {model_path.relative_to(_root)}")
print(pipeline)

Modelo cargado: models/06_svm_model.joblib
Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'rest_bp', 'chol',
                                                   'max_hr', 'old_peak']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                

## Importancia de Características

Se usa **Permutation Importance**: mide cuánto cae el F1 cuando se mezclan aleatoriamente los valores de cada feature. Al aplicarlo sobre el pipeline completo, la importancia se calcula en el espacio de las variables originales (antes del encoding), lo que facilita la interpretación clínica.

In [24]:
# Permutation importance sobre el conjunto de test
result = permutation_importance(
    pipeline,
    X_test,
    y_test,
    scoring="f1",
    n_repeats=N_REPEATS,
    random_state=SEED,
    n_jobs=-1,
)

feature_names = num_cols + cat_cols
importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }
).sort_values("importance_mean", ascending=False)

print(importance_df.to_string(index=False))

   feature  importance_mean  importance_std
  old_peak         0.070822        0.015630
      thal         0.056114        0.027581
        ca         0.054676        0.023644
chest_pain         0.038676        0.021513
     slope         0.032755        0.013208
    max_hr         0.017837        0.013913
       age         0.005864        0.014619
   rest_bp         0.005837        0.015849
       sex         0.004293        0.012474
  rest_ecg         0.003008        0.006015
      chol         0.001475        0.006726
       fbs         0.000000        0.000000


In [25]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    importance_df["feature"],
    importance_df["importance_mean"],
    xerr=importance_df["importance_std"],
    color="steelblue",
    ecolor="black",
    capsize=4,
)
ax.set_xlabel("Caída en F1 (mayor = más importante)")
ax.set_title("Permutation Importance — SVM")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "permutation_importance.png", dpi=72)
plt.close(fig)
print("Gráfica guardada.")

Gráfica guardada.


## 📊 Analysis of Results and Conclusions 

Description of the results obtained and if there are conclusions that can be drawn from them.

The analysis of results must be related to the description of the task.

**Note:** An analysis of results does not necessarily lead to conclusions, but to ideas or proposals for future work


## Curva de Aprendizaje y Escalabilidad

Análisis de cómo evoluciona el rendimiento del modelo a medida que aumenta el tamaño del conjunto de entrenamiento, y métricas de tiempo de entrenamiento para evaluar escalabilidad.

In [ ]:
train_sizes, train_scores, val_scores, fit_times, _ = learning_curve(
    pipeline,
    X_clean,
    y,
    cv=CV_FOLDS,
    scoring="f1",
    train_sizes=np.linspace(0.1, 1.0, 10),
    return_times=True,
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
val_mean = val_scores.mean(axis=1)
val_std = val_scores.std(axis=1)
fit_mean = fit_times.mean(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Curva de aprendizaje
axes[0].fill_between(
    train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color="blue"
)
axes[0].fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color="orange")
axes[0].plot(train_sizes, train_mean, "o-", color="blue", label="Train F1")
axes[0].plot(train_sizes, val_mean, "o-", color="orange", label="Validation F1")
axes[0].set_xlabel("Tamaño de entrenamiento")
axes[0].set_ylabel("F1 Score")
axes[0].set_title("Curva de Aprendizaje")
axes[0].legend()

# Escalabilidad
axes[1].plot(train_sizes, fit_mean, "o-", color="green")
axes[1].set_xlabel("Tamaño de entrenamiento")
axes[1].set_ylabel("Tiempo de entrenamiento (s)")
axes[1].set_title("Escalabilidad del Modelo")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "learning_curve_interpretacion.png", dpi=72)
plt.close(fig)
print("Gráfica guardada.")

Gráfica guardada.


## Análisis de Errores

Se analiza qué tipo de errores comete el modelo: falsos positivos (FP) y falsos negativos (FN), y qué características tienen los registros mal clasificados.

In [ ]:
y_pred = pipeline.predict(X_test)
y_prob = pipeline.predict_proba(X_test)[:, 1]

print("Métricas en test:")
print(f"  Accuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"  F1       : {f1_score(y_test, y_pred):.4f}")
print(f"  AUC-ROC  : {roc_auc_score(y_test, y_prob):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=["Sin enfermedad", "Con enfermedad"]))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Sin enfermedad", "Con enfermedad"]
)
disp.plot(ax=ax, colorbar=False, cmap="Blues")
ax.set_title("Matriz de Confusión — SVM")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "confusion_matrix_interpretacion.png", dpi=72)
plt.close(fig)
print("Matriz de confusión guardada.")

Métricas en test:
  Accuracy : 0.8642
  F1       : 0.8571
  AUC-ROC  : 0.9296

                precision    recall  f1-score   support

Sin enfermedad       0.93      0.82      0.87        45
Con enfermedad       0.80      0.92      0.86        36

      accuracy                           0.86        81
     macro avg       0.86      0.87      0.86        81
  weighted avg       0.87      0.86      0.86        81

Matriz de confusión guardada.


In [28]:
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

errors_df = X_test_reset.copy()
errors_df["y_real"] = y_test_reset
errors_df["y_pred"] = y_pred
errors_df["error_type"] = "Correcto"
errors_df.loc[(errors_df["y_real"] == 0) & (errors_df["y_pred"] == 1), "error_type"] = (
    "Falso Positivo"
)
errors_df.loc[(errors_df["y_real"] == 1) & (errors_df["y_pred"] == 0), "error_type"] = (
    "Falso Negativo"
)

print("Distribución de errores:")
print(errors_df["error_type"].value_counts())
print()

fp = errors_df[errors_df["error_type"] == "Falso Positivo"]
fn = errors_df[errors_df["error_type"] == "Falso Negativo"]

print(f"Falsos Positivos ({len(fp)}): paciente sano clasificado como enfermo")
print(f"Falsos Negativos ({len(fn)}): paciente enfermo clasificado como sano")
print()
print("Estadísticas FP vs FN en variables numéricas:")
print(errors_df.groupby("error_type")[num_cols].mean().round(2))

Distribución de errores:
error_type
Correcto          70
Falso Positivo     8
Falso Negativo     3
Name: count, dtype: int64

Falsos Positivos (8): paciente sano clasificado como enfermo
Falsos Negativos (3): paciente enfermo clasificado como sano

Estadísticas FP vs FN en variables numéricas:
                  age  rest_bp    chol  max_hr  old_peak
error_type                                              
Correcto        53.86   130.09  236.23  150.44      1.40
Falso Negativo  53.00   119.33  206.00  149.00      1.07
Falso Positivo  60.25   129.25  242.62  150.12      0.92


## Consecuencias de las Malas Predicciones

En el contexto clínico de diagnóstico de enfermedad cardíaca:

| Error | Descripción | Consecuencia |
|---|---|---|
| **Falso Positivo (8)** | Paciente sano clasificado como enfermo | Estrés innecesario, exámenes adicionales, costos médicos |
| **Falso Negativo (3)** | Paciente enfermo clasificado como sano | **Alto riesgo**: el paciente no recibe tratamiento oportuno, puede derivar en eventos cardíacos graves |

**Conclusión:** En medicina, los Falsos Negativos son más críticos que los FP. El modelo actual tiene 3 FN vs 8 FP, lo que indica que es más conservador (prefiere alertar antes que dejar pasar casos). En producción se debería evaluar reducir el umbral de decisión (< 0.5) para minimizar FN a costa de más FP.

In [29]:
# ¿Los errores se concentran en valores extremos?
fig, axes = plt.subplots(1, len(num_cols), figsize=(16, 4))

for i, col in enumerate(num_cols):
    bp = axes[i].boxplot(
        [
            errors_df[errors_df["error_type"] == "Correcto"][col].dropna(),
            errors_df[errors_df["error_type"] == "Falso Positivo"][col].dropna(),
            errors_df[errors_df["error_type"] == "Falso Negativo"][col].dropna(),
        ],
        patch_artist=True,
    )
    axes[i].set_xticks([1, 2, 3])
    axes[i].set_xticklabels(["OK", "FP", "FN"])
    axes[i].set_title(col)

plt.suptitle("Distribución de variables numéricas por tipo de error")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "error_analysis_boxplot.png", dpi=72)
plt.close(fig)
print("Gráfica guardada.")

Gráfica guardada.


In [30]:
# ¿Clase desbalanceada?
class_dist = y.value_counts()
print("Distribución de clases en el dataset completo:")
print(class_dist)
print(f"\nRatio: {class_dist[0] / class_dist[1]:.2f} (sin enfermedad / con enfermedad)")

fig, ax = plt.subplots(figsize=(4, 3))
class_dist.plot(kind="bar", ax=ax, color=["steelblue", "salmon"], rot=0)
ax.set_xticklabels(["Sin enfermedad", "Con enfermedad"])
ax.set_ylabel("Cantidad")
ax.set_title("Balance de clases")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "class_balance.png", dpi=72)
plt.close(fig)
print("Gráfica guardada.")

Distribución de clases en el dataset completo:
disease
0    224
1    180
Name: count, dtype: int64

Ratio: 1.24 (sin enfermedad / con enfermedad)
Gráfica guardada.


##  Causas de los Errores

### Análisis por factor

| Factor | Hallazgo | Impacto |
|---|---|---|
| **Outliers** | Los FP tienen mayor edad (60 vs 54) y colesterol (243 vs 236). Valores extremos en pacientes mayores confunden al modelo | Moderado |
| **Clase desbalanceada** | Ratio 1.24:1 (224 sanos vs 180 enfermos). Desbalance leve, no crítico | Bajo |
| **Errores en datos** | Dataset original tenía valores inválidos (sex='2345', disease='fsg') ya corregidos en

## Conclusiones

1. Las variables más predictivas son `old_peak`, `thal` y `ca` — coherente con la literatura clínica cardiovascular.
2. El modelo comete más FP (8) que FN (3), lo que en contexto médico es preferible pero el umbral de decisión debería ajustarse.
3. El desbalance de clases es leve (1.24:1) y no justifica técnicas de oversampling por sí solo.
4. El tamaño reducido del dataset (404 registros) es el principal limitante para la generalización.
